In [ ]:
from datasets import load_dataset

# Algorithm-generated
ds_algorithm = load_dataset(
    "Kevin355/Who_and_When",
    "Algorithm-Generated"
)

# Hand-crafted
ds_handcrafted = load_dataset(
    "Kevin355/Who_and_When",
    "Hand-Crafted"
)

print(ds_algorithm)
print(ds_handcrafted)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Create figures folder if it does not already exist
FIGURE_DIR = Path("figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Figures will be saved to: {FIGURE_DIR.resolve()}")

In [ ]:
#Inspecting the data splits and fields

datasets = {
    "Algorithm-Generated": ds_algorithm,
    "Hand-Crafted": ds_handcrafted
}

for category, dataset_dict in datasets.items():
    print(f"\n{'='*60}")
    print(category)

    for split, dataset in dataset_dict.items():
        print(f"\nSplit: {split}")
        print(f"Total records: {len(dataset)}")
        print(f"Columns: {dataset.column_names}")
        print(f"Features: {dataset.features}")

        if len(dataset):
            print("\nFirst record:")
            for field, value in dataset[0].items():
                print(f"{field}: {str(value)[:300]}")

In [ ]:
#Validate missing values and duplicates
validation_results = []

for category, dataset_dict in datasets.items():
    for split, dataset in dataset_dict.items():

        df = dataset.to_pandas()

        result = {
            "category": category,
            "split": split,
            "total_records": len(df),
            "missing_cells": int(df.isna().sum().sum())
        }

        for col in ["id", "trace_id", "sample_id"]:
            if col in df.columns:
                result["duplicate_ids"] = int(
                    df[col].duplicated().sum()
                )
                break

        validation_results.append(result)

validation_df = pd.DataFrame(validation_results)

display(validation_df)

Path("results").mkdir(exist_ok=True)

validation_df.to_csv(
    "results/dataset_validation_summary.csv",
    index=False
)

In [ ]:
# Examine how traces are structured
for category, dataset_dict in datasets.items():

    dataset = dataset_dict["train"]

    print(f"\n{'='*60}")
    print(category)

    sample = dataset[0]

    print("Mistake agent:", sample["mistake_agent"])
    print("Mistake step:", sample["mistake_step"])
    print("Mistake reason:", sample["mistake_reason"])

    print("\nTRACE HISTORY:")

    for i, message in enumerate(sample["history"][:10]):

        print(f"\n--- Message {i} ---")

        print("Role:", message.get("role"))
        print("Name:", message.get("name", "Not provided"))
        print("Content:", str(message.get("content", ""))[:600])

In [ ]:
# Identify the missing values
import pandas as pd

for category, dataset_dict in datasets.items():
    df = dataset_dict["train"].to_pandas()

    print(f"\n{'='*60}")
    print(category)

    missing = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": (
            df.isna().mean() * 100
        ).round(2)
    })

    display(missing)

    print("\nRows with missing values:")
    display(
        df.loc[df.isna().any(axis=1),
               ["question_ID", "mistake_agent",
                "mistake_step", "mistake_reason"]
        ].head(10)
    )

In [ ]:
# validate step indexing against the trace
import pandas as pd

step_results = []

for category, dataset_dict in datasets.items():
    dataset = dataset_dict["train"]

    for record in dataset:
        history = record["history"]
        mistake_step = int(record["mistake_step"])
        mistake_agent = record["mistake_agent"]

        # Locate messages associated with the agent
        agent_positions = []

        for i, message in enumerate(history):
            name = message.get("name") or ""
            role = message.get("role") or ""

            if (
                name == mistake_agent
                or role == mistake_agent
            ):
                agent_positions.append(i)

        step_results.append({
            "category": category,
            "question_ID": record["question_ID"],
            "mistake_agent": mistake_agent,
            "mistake_step": mistake_step,
            "history_length": len(history),
            "within_message_range": (
                0 <= mistake_step < len(history)
            ),
            "agent_positions": agent_positions,
            "direct_agent_match": (
                mistake_step in agent_positions
            )
        })

step_df = pd.DataFrame(step_results)

display(
    step_df.groupby("category").agg(
        total=("question_ID", "count"),
        within_range=("within_message_range", "sum"),
        direct_agent_matches=("direct_agent_match", "sum")
    )
)

step_df.to_csv(
    "results/step_validation.csv",
    index=False
)

In [ ]:
# Inspect mismatched annotations
mismatches = step_df[
    ~step_df["direct_agent_match"]
]

print("Records without direct agent match:", len(mismatches))

display(
    mismatches[
        [
            "category",
            "question_ID",
            "mistake_agent",
            "mistake_step",
            "history_length",
            "agent_positions"
        ]
    ].head(20)
)

In [ ]:
# Investigating the Mismatches
import pandas as pd
import re

def normalize_agent(name):
    if name is None:
        return ""
    return re.sub(r"[^a-z0-9]", "", str(name).lower())

def message_agents(message):
    agents = []

    name = message.get("name")
    role = message.get("role", "")

    if name:
        agents.append(normalize_agent(name))

    # Exact speaker or Orchestrator subrole
    if role:
        agents.append(normalize_agent(role.split("(")[0].strip()))

    return agents

review = []

for category, dataset_dict in datasets.items():
    for record in dataset_dict["train"]:

        history = record["history"]
        step = int(record["mistake_step"])
        agent = normalize_agent(record["mistake_agent"])

        positions = [
            i for i, msg in enumerate(history)
            if agent in message_agents(msg)
        ]

        direct_match = step in positions

        review.append({
            "category": category,
            "question_ID": record["question_ID"],
            "mistake_agent": record["mistake_agent"],
            "mistake_step": step,
            "history_length": len(history),
            "within_range": 0 <= step < len(history),
            "normalized_agent_match": direct_match,
            "agent_positions": positions,
            "previous_role": (
                history[step-1].get("role")
                if 0 <= step-1 < len(history)
                else None
            ),
            "step_role": (
                history[step].get("role")
                if 0 <= step < len(history)
                else None
            )
        })

review_df = pd.DataFrame(review)

display(
    review_df.groupby("category").agg(
        total=("question_ID", "count"),
        in_range=("within_range", "sum"),
        normalized_matches=("normalized_agent_match", "sum")
    )
)

print("\nRecords requiring manual review:")

display(
    review_df[
        (~review_df["within_range"]) |
        (~review_df["normalized_agent_match"])
    ]
)

review_df.to_csv(
    "results/annotation_alignment_review.csv",
    index=False
)

In [ ]:
 # Inspecting the out of range records
 for category, dataset_dict in datasets.items():
    for record in dataset_dict["train"]:

        history = record["history"]
        step = int(record["mistake_step"])

        if step >= len(history):

            print("\n" + "=" * 70)
            print("Category:", category)
            print("ID:", record["question_ID"])
            print("Mistake agent:", record["mistake_agent"])
            print("Mistake step:", step)
            print("History length:", len(history))
            print("Mistake reason:", record["mistake_reason"])

            print("\nAvailable trace messages:")

            for i, msg in enumerate(history):
                print(
                    i,
                    msg.get("role"),
                    str(msg.get("content", ""))[:250]
                )

In [ ]:
# the annotations review report
from pathlib import Path
import pandas as pd

Path("results").mkdir(exist_ok=True)

unresolved = review_df[
    (~review_df["within_range"]) |
    (~review_df["normalized_agent_match"])
].copy()

unresolved["review_status"] = "pending"
unresolved["review_reason"] = unresolved.apply(
    lambda row: (
        "Out-of-range mistake step"
        if not row["within_range"]
        else "Agent does not match indexed message"
    ),
    axis=1
)

unresolved["manual_notes"] = ""

unresolved.to_csv(
    "results/annotation_review.csv",
    index=False
)

print("Total records requiring review:", len(unresolved))

display(
    unresolved[
        [
            "category",
            "question_ID",
            "mistake_agent",
            "mistake_step",
            "history_length",
            "review_reason"
        ]
    ]
)

In [ ]:
# Checking for offsets of the index
offset_results = []

for _, row in review_df.iterrows():

    step = row["mistake_step"]
    positions = row["agent_positions"]

    offset_results.append({
        "category": row["category"],
        "question_ID": row["question_ID"],
        "exact_match": step in positions,
        "minus_one_match": (step - 1) in positions,
        "plus_one_match": (step + 1) in positions,
        "within_range": row["within_range"]
    })

offset_df = pd.DataFrame(offset_results)

display(
    offset_df.groupby("category")[
        [
            "exact_match",
            "minus_one_match",
            "plus_one_match"
        ]
    ].sum()
)

In [ ]:
# Identify unresolved records directly from review_df
unresolved = review_df[
    ~review_df["normalized_agent_match"]
][["category", "question_ID"]].copy()

# Match using both category and question_ID
unresolved_offsets = offset_df.merge(
    unresolved,
    on=["category", "question_ID"],
    how="inner",
    validate="one_to_one"
)

print("Total unresolved records:", len(unresolved_offsets))

display(
    unresolved_offsets[
        [
            "category",
            "question_ID",
            "exact_match",
            "minus_one_match",
            "plus_one_match",
            "within_range"
        ]
    ]
)

print("\nOffset matches among unresolved records:")

display(
    unresolved_offsets.groupby("category")[
        ["minus_one_match", "plus_one_match"]
    ].sum()
)

In [ ]:
# Validation Report
summary = {
    "total_records": 184,
    "algorithm_generated_records": 126,
    "handcrafted_records": 58,
    "numeric_mistake_steps": 184,
    "in_range_steps": 181,
    "normalized_agent_matches": 175,
    "records_requiring_review": 9,
    "out_of_range_records": 3,
    "indexing_convention_verified": False,
    "baseline_reproduced": False
}

import json
from pathlib import Path

Path("results").mkdir(exist_ok=True)

with open(
    "results/dataset_validation.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(summary, f, indent=4)

print(json.dumps(summary, indent=4))